### Project

In [31]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [32]:
import gradio as gr
import pandas as pd

In [33]:
def load_file(file):

    if file is None:
        return None

    # Handles newer Gradio versions
    if isinstance(file, str):
        file_path = file
    else:
        file_path = file.name


    if file_path.endswith(".csv"):
        df = pd.read_csv(file_path)

    elif file_path.endswith(".xlsx"):
        df = pd.read_excel(file_path)

    else:
        return None

    return df

In [34]:
def analyze_dataset(file):

    if file is None:
        return "Please upload a file"


    if file.name.endswith(".csv"):
        df = pd.read_csv(file.name)

    elif file.name.endswith(".xlsx"):
        df = pd.read_excel(file.name)

    else:
        return "Unsupported file type"


    missing = df.isnull().sum().to_string()

    datatype = df.dtypes.to_string()


    summary = f"""
DATASET SUMMARY
----------------

Rows: {df.shape[0]}
Columns: {df.shape[1]}

Columns:
{list(df.columns)}


DATA QUALITY REPORT
-------------------

Missing Values:

{missing}


Duplicate Rows:

{df.duplicated().sum()}


DATA TYPES
----------

{datatype}

"""

    return summary

In [35]:
def statistical_analysis(file):

    if file is None:
        return "Please upload a file"


    if file.name.endswith(".csv"):
        df = pd.read_csv(file.name)

    elif file.name.endswith(".xlsx"):
        df = pd.read_excel(file.name)

    else:
        return "Unsupported file type"


    # Detect numerical columns automatically
    numerical_columns = df.select_dtypes(
        include=["int64", "float64"]
    ).columns


    if len(numerical_columns) == 0:
        return "No numerical columns found"


    result = "STATISTICAL ANALYSIS\n"
    result += "--------------------\n\n"


    for column in numerical_columns:

        result += f"{column}\n"
        result += "----------------\n"

        result += f"Mean: {df[column].mean():.2f}\n"
        result += f"Median: {df[column].median():.2f}\n"
        result += f"Standard Deviation: {df[column].std():.2f}\n"
        result += f"Minimum: {df[column].min():.2f}\n"
        result += f"Maximum: {df[column].max():.2f}\n\n"


    return result

In [37]:
def generate_graph(file):

    df = load_file(file)

    if df is None:
        return None


    numerical = df.select_dtypes(
        include=["number"]
    ).columns


    categorical = df.select_dtypes(
        include=["object"]
    ).columns


    datetime_cols = df.select_dtypes(
        include=["datetime"]
    ).columns


    print("Numerical:", list(numerical))
    print("Categorical:", list(categorical))


    if len(categorical) >= 1 and len(numerical) >= 1:

        fig = px.bar(
            df,
            x=categorical[0],
            y=numerical[0],
            title=f"{numerical[0]} by {categorical[0]}"
        )

        return fig


    elif len(datetime_cols) >= 1 and len(numerical) >= 1:

        fig = px.line(
            df,
            x=datetime_cols[0],
            y=numerical[0],
            title=f"{numerical[0]} Trend"
        )

        return fig


    elif len(numerical) >= 2:

        fig = px.scatter(
            df,
            x=numerical[0],
            y=numerical[1],
            title=f"{numerical[0]} vs {numerical[1]}"
        )

        return fig


    elif len(numerical) == 1:

        fig = px.histogram(
            df,
            x=numerical[0],
            title=f"Distribution of {numerical[0]}"
        )

        return fig


    return None

In [ ]:
!pip install gradio pandas openpyxl plotly

In [38]:
import gradio as gr
import pandas as pd
import plotly.express as px


# -----------------------------
# Function to load dataset
# -----------------------------

def load_file(file):

    if file is None:
        return None

    if file.name.endswith(".csv"):
        df = pd.read_csv(file.name)

    elif file.name.endswith(".xlsx"):
        df = pd.read_excel(file.name)

    else:
        return None

    return df



# -----------------------------
# Cleaning Report
# -----------------------------

def analyze_dataset(file):

    df = load_file(file)

    if df is None:
        return "Unsupported file type"


    missing = df.isnull().sum().to_string()

    datatypes = df.dtypes.to_string()


    report = f"""
DATASET SUMMARY
----------------

Rows: {df.shape[0]}
Columns: {df.shape[1]}


COLUMNS
-------

{list(df.columns)}



DATA QUALITY REPORT
-------------------

Missing Values:

{missing}



Duplicate Rows:

{df.duplicated().sum()}



DATA TYPES
----------

{datatypes}

"""


    return report



# -----------------------------
# Statistical Analysis
# -----------------------------

def statistical_analysis(file):

    df = load_file(file)

    if df is None:
        return "Unsupported file type"


    numerical_columns = df.select_dtypes(
        include=["int64","float64"]
    ).columns


    if len(numerical_columns)==0:
        return "No numerical columns found"


    result = """
STATISTICAL ANALYSIS
--------------------

"""


    for column in numerical_columns:

        result += f"""
{column}
----------------

Mean:
{df[column].mean():.2f}

Median:
{df[column].median():.2f}

Standard Deviation:
{df[column].std():.2f}

Minimum:
{df[column].min():.2f}

Maximum:
{df[column].max():.2f}


"""


    return result



# -----------------------------
# Automatic Graph Generation
# -----------------------------

def generate_graph(file):

    df = load_file(file)

    if df is None:
        return None


    numerical = df.select_dtypes(
        include=["int64","float64"]
    ).columns


    categorical = df.select_dtypes(
        include=["object"]
    ).columns



    # Two numerical columns
    if len(numerical) >= 2:

        fig = px.scatter(
            df,
            x=numerical[0],
            y=numerical[1],
            title="Relationship Between Variables"
        )

        return fig



    # Category + Numerical
    elif len(categorical) >= 1 and len(numerical) >= 1:

        fig = px.bar(
            df,
            x=categorical[0],
            y=numerical[0],
            title="Category Comparison"
        )

        return fig



    # Single numerical column
    elif len(numerical) >= 1:

        fig = px.histogram(
            df,
            x=numerical[0],
            title="Data Distribution"
        )

        return fig


    else:

        return None




# -----------------------------
# Gradio Interface
# -----------------------------

with gr.Blocks(
    theme=gr.themes.Soft()
) as app:


    gr.Markdown(
        "# 📊 Automated Data Analyzer"
    )


    file_upload = gr.File(
        label="Upload Dataset (CSV/XLSX)"
    )


    analyze_button = gr.Button(
        "Analyze Dataset"
    )



    with gr.Tab("Cleaning Report"):

        cleaning_output = gr.Textbox(
            lines=25,
            label="Data Quality Report"
        )



    with gr.Tab("Statistics"):

        statistics_output = gr.Textbox(
            lines=25,
            label="Statistical Analysis"
        )



    with gr.Tab("Visualization"):

        graph_output = gr.Plot(
            label="Recommended Visualization"
        )



    analyze_button.click(
        analyze_dataset,
        inputs=file_upload,
        outputs=cleaning_output
    )


    analyze_button.click(
        statistical_analysis,
        inputs=file_upload,
        outputs=statistics_output
    )


    analyze_button.click(
        generate_graph,
        inputs=file_upload,
        outputs=graph_output
    )



app.launch(theme=gr.themes.Soft())

C:\Users\deepi\AppData\Local\Temp\ipykernel_22768\2849016200.py:218: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.



* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


C:\Users\deepi\AppData\Roaming\Python\Python313\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning:

'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.

C:\Users\deepi\AppData\Roaming\Python\Python313\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning:

'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.

C:\Users\deepi\AppData\Roaming\Python\Python313\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning:

'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.

C:\Users\deepi\AppData\Roaming\Python\Python313\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning:

'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.

C:\Users\deepi\AppData\Roaming\Python\Python313\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning:

'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Us